In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from db_connection import fetch_data

# Visual configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Fetch data directly from MySQL
query = "SELECT * FROM sbi_branch_metrics;"
df = fetch_data(query)

print(f"Data retrieved successfully. Total Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

In [ ]:
# Verify pre vs post split
df["merger_phase"] = np.where(df["fiscal_year"] <= 2017, "Pre-Merger", "Post-Merger")

metrics = ["gross_npa_pct", "net_npa_pct", "cost_to_income_pct", "roa_pct", "car_crar_pct"]
summary_table = df.groupby("merger_phase")[metrics].agg(["mean", "std", "min", "max"]).round(2)
display(summary_table)

In [ ]:
print("=== STATISTICAL HYPOTHESIS TESTING (Pre vs. Post Merger) ===")

pre_data = df[df["merger_phase"] == "Pre-Merger"]
post_data = df[df["merger_phase"] == "Post-Merger"]

for metric in ["gross_npa_pct", "roa_pct", "cost_to_income_pct", "car_crar_pct"]:
    t_stat, p_val = stats.ttest_ind(pre_data[metric], post_data[metric], equal_var=False)
    sig = "Statistically Significant (p < 0.05)" if p_val < 0.05 else "Not Significant"
    print(f"Metric: {metric:<20} | t-statistic: {t_stat:>8.3f} | p-value: {p_val:.4e} -> {sig}")

In [ ]:
plt.figure(figsize=(11, 5))
yearly_trend = df.groupby(["fiscal_year", "merger_phase"])["gross_npa_pct"].mean().reset_index()

sns.lineplot(data=yearly_trend, x="fiscal_year", y="gross_npa_pct", marker="o", linewidth=2.5, color="#c0392b")
plt.axvline(x=2017.5, color="black", linestyle="--", linewidth=1.5, label="Merger Cutoff (FY2017)")
plt.title("SBI Gross NPA (%) Trajectory (FY2014 - FY2021)", fontsize=13, weight="bold")
plt.xlabel("Fiscal Year")
plt.ylabel("Mean Gross NPA (%)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x="roa_pct", hue="merger_phase", fill=True, common_norm=False, palette=["#e74c3c", "#27ae60"], alpha=0.4)
plt.axvline(x=0, color="gray", linestyle="--", linewidth=1)
plt.title("Return on Assets (RoA %) Distribution: Pre-Merger vs. Post-Merger", fontsize=13, weight="bold")
plt.xlabel("Return on Assets (%)")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x="branch_tier", y="cost_to_income_pct", hue="merger_phase", palette="Blues_r", errorbar=None)
plt.title("Operating Efficiency (Cost-to-Income %) Across Branch Tiers", fontsize=13, weight="bold")
plt.xlabel("Branch Tier")
plt.ylabel("Mean Cost-to-Income (%)")
plt.legend(title="Phase")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
correlation_cols = ["gross_npa_pct", "net_npa_pct", "cost_to_income_pct", "roa_pct", "casa_ratio_pct", "car_crar_pct"]
corr = df[correlation_cols].corr()

sns.heatmap(corr, annot=True, cmap="vlag", fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Correlation Matrix of Financial & Risk Indicators", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()